In [2]:
import argparse
import numpy as np


def total_energy(spins: np.ndarray, J: float = 1.0) -> float:
    """Total energy with periodic boundary conditions."""
    # Count each bond once: right + down neighbors
    return -J * np.sum(spins * (np.roll(spins, 1, axis=0) + np.roll(spins, 1, axis=1)))


def delta_energy(spins: np.ndarray, i: int, j: int, J: float = 1.0) -> float:
    """Energy change if spin (i,j) is flipped."""
    L = spins.shape[0]
    s = spins[i, j]
    nn = (
        spins[(i + 1) % L, j]
        + spins[(i - 1) % L, j]
        + spins[i, (j + 1) % L]
        + spins[i, (j - 1) % L]
    )
    return 2.0 * J * s * nn


def metropolis_sweep(spins: np.ndarray, beta: float, J: float = 1.0, rng=None) -> None:
    """One Monte Carlo sweep: L*L attempted flips."""
    if rng is None:
        rng = np.random.default_rng()
    L = spins.shape[0]
    for _ in range(L * L):
        i = rng.integers(0, L)
        j = rng.integers(0, L)
        dE = delta_energy(spins, i, j, J)
        if dE <= 0 or rng.random() < np.exp(-beta * dE):
            spins[i, j] *= -1


def main():
    parser = argparse.ArgumentParser(description="Simple 2D Ising model (Metropolis).")
    parser.add_argument("--L", type=int, default=32, help="Lattice size (LxL)")
    parser.add_argument("--T", type=float, default=2.3, help="Temperature")
    parser.add_argument("--J", type=float, default=1.0, help="Coupling constant")
    parser.add_argument("--sweeps", type=int, default=5000, help="Total sweeps")
    parser.add_argument("--burnin", type=int, default=1000, help="Burn-in sweeps")
    parser.add_argument("--seed", type=int, default=0, help="RNG seed")
    args = parser.parse_args()

    rng = np.random.default_rng(args.seed)
    beta = 1.0 / args.T

    # Random initial state: +/-1
    spins = rng.choice([-1, 1], size=(args.L, args.L))

    # Burn-in
    for _ in range(args.burnin):
        metropolis_sweep(spins, beta, args.J, rng)

    # Sampling
    e_samples = []
    m_samples = []
    for _ in range(args.sweeps):
        metropolis_sweep(spins, beta, args.J, rng)
        e_samples.append(total_energy(spins, args.J) / (args.L * args.L))
        m_samples.append(np.mean(spins))

    print(f"L={args.L}, T={args.T}, J={args.J}")
    print(f"<E>/spin = {np.mean(e_samples):.6f}")
    print(f"<|M|>    = {np.mean(np.abs(m_samples)):.6f}")


if __name__ == "__main__":
    main()

usage: ipykernel_launcher.py [-h] [--L L] [--T T] [--J J] [--sweeps SWEEPS]
                             [--burnin BURNIN] [--seed SEED]
ipykernel_launcher.py: error: unrecognized arguments: --f=/home/codespace/.local/share/jupyter/runtime/kernel-v35722344477cfdb543f2957ede69f2824a8ca4e74.json


SystemExit: 2